# SWAP-Stress: AMSR vegetation optical depth

**AMSR-E/AMSR2 LPDR v3** vegetation optical depth (VOD) at 10.7 GHz is a
microwave proxy for vegetation water content and biomass. LPDR v3 is daily and
global on a ~25 km EASE-Grid (EPSG:3410) from 2002 through 2022, with separate
ascending and descending passes.

VOD is the one covariate that does **not** come through Earth Engine.
`swapstress.features.amsr_extract` samples the nearest grid cell at each training
site and reduces the daily record to seasonal statistics — 5 periods (winter,
spring, summer, autumn, annual) × 2 statistics (mean, stddev) × 2 passes =
**20 features** per site. `build_unified_table(..., amsr_vod_path=...)` merges
the resulting parquet into the training table.

1. Inside one AMSR NetCDF file
2. The daily VOD record at a ReESH site
3. The season definitions, and the site's precomputed climatology
4. The climatology parquet across all sites
5. Feature distributions
6. Spatial pattern of summer VOD

In [ ]:
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from swapstress.features.build_training_table import default_output_path

# ---------------------------------------------------------------------------
# Paths you may have to change. AMSR is not sampled through Earth Engine, so it
# has no entry in the source registry: the NetCDF directory and the climatology
# parquet are supplied here rather than resolved. The training table path does
# come from the library.
# ---------------------------------------------------------------------------
DATA_ROOT = os.environ.get("SWAPSTRESS_DATA_ROOT", "/nas/soils")
SCALE = "9km_global"

AMSR_DIR = os.path.join(DATA_ROOT, "swapstress", "amsr")
VOD_CLIM_PATH = os.path.join(
    DATA_ROOT, "swapstress", "training", "amsr_vod_climatology.parquet"
)

# Written by stage 02; this is the same default the stage uses.
SITES_PARQUET = default_output_path(DATA_ROOT, SCALE, embeddings=False)

SITE_ID = "US-CDM"  # a ReESH station
SITE_KEY = SITE_ID.replace("-", "_")  # sample_ids use underscores

OUT_DIR = os.path.join("notebooks", "_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

print("AMSR_DIR:      ", AMSR_DIR)
print("SITES_PARQUET: ", SITES_PARQUET)
print(
    "VOD_CLIM_PATH: ",
    VOD_CLIM_PATH,
    "" if os.path.exists(VOD_CLIM_PATH) else " [not built yet — see section 0]",
)

## 0) Producing the climatology

The extractor is not one of the numbered stages — it runs once, ahead of stage
02, and its output is passed in as `amsr_vod_path`:

```bash
uv run python -m swapstress.features.amsr_extract \
    --amsr-dir       /nas/soils/swapstress/amsr \
    --sites-parquet  /nas/soils/swapstress/training/obs_level_training_9km_global.parquet \
    --output         /nas/soils/swapstress/training/amsr_vod_climatology.parquet
```

It reads twenty years of daily global grids, so it takes a while. The rest of
this notebook reads what it already wrote.

## 1) Inside one AMSR file

One NetCDF per year per orbit pass, named
`AMSR-E-2_LPDRv3_Y{year}_{A,D}.nc4` — `A` is ascending (afternoon equator
crossing), `D` is descending (early morning). Coordinates are EASE-Grid metres.

In [ ]:
sample_file = os.path.join(AMSR_DIR, "AMSR-E-2_LPDRv3_Y2015_A.nc4")

with xr.open_dataset(sample_file) as ds:
    print("Dimensions:", dict(ds.sizes))
    print("Variables: ", list(ds.data_vars))
    print(f"x range:   {ds.x.values.min():,.0f} to {ds.x.values.max():,.0f} m")
    print(f"y range:   {ds.y.values.min():,.0f} to {ds.y.values.max():,.0f} m")
    print(
        f"time:      {pd.Timestamp(ds.time.values[0]).date()} to "
        f"{pd.Timestamp(ds.time.values[-1]).date()}  ({len(ds.time)} steps)"
    )
    print("VOD attrs:", dict(ds["VOD"].attrs))

    vod_slice = ds["VOD"].isel(time=180).values

valid = vod_slice[(vod_slice >= 0) & (vod_slice <= 3)]
print(
    f"\nVOD on day 181: shape={vod_slice.shape}, valid={len(valid):,} px, "
    f"range=[{valid.min():.3f}, {valid.max():.3f}], median={np.median(valid):.3f}"
)

## 2) The daily record at one site

The extractor reduces the daily record to seasonal statistics and never returns
the series itself, so this section reads the annual files directly. The
geolocation is still the library's: `_reproject_sites` and
`_find_nearest_indices` are the same helpers
`extract_amsr_vod_climatology` uses internally, so the pixel picked here is the
pixel the features were computed from. Only the file walk and the plot are local
to the notebook.

In [ ]:
from swapstress.features.amsr_extract import (
    _find_nearest_indices,
    _load_sites,
    _reproject_sites,
)

sites = _load_sites(SITES_PARQUET)
site = sites[sites["sample_id"].astype(str).str.startswith(f"reesh_{SITE_KEY}")].iloc[0]
site_lat, site_lon = float(site["lat"]), float(site["lon"])
print(f"{SITE_ID}: lat={site_lat:.4f}, lon={site_lon:.4f}")

site_x, site_y = _reproject_sites(np.array([site_lat]), np.array([site_lon]))
print(f"EASE-Grid (EPSG:3410): x={site_x[0]:,.0f} m, y={site_y[0]:,.0f} m")

In [ ]:
records = []
grid_yi = grid_xi = None

for year in range(2002, 2023):
    for code, pass_name in {"A": "asc", "D": "desc"}.items():
        fpath = os.path.join(AMSR_DIR, f"AMSR-E-2_LPDRv3_Y{year}_{code}.nc4")
        if not os.path.exists(fpath):
            continue

        with xr.open_dataset(fpath) as ds:
            if grid_xi is None:
                gx, gy = ds.x.values, ds.y.values
                y_descending = gy[0] > gy[-1]
                gy_ascending = gy[::-1] if y_descending else gy
                grid_xi = _find_nearest_indices(gx, site_x)[0]
                yi = _find_nearest_indices(gy_ascending, site_y)[0]
                grid_yi = (len(gy) - 1 - yi) if y_descending else yi

            vod = ds["VOD"].values[:, grid_yi, grid_xi].astype(float)
            times = pd.DatetimeIndex(ds.time.values)

        vod[(vod < 0) | (vod > 3)] = np.nan  # the extractor's valid range
        records.append(pd.DataFrame({"date": times, "pass": pass_name, "vod": vod}))

ts = pd.concat(records, ignore_index=True)
ts_wide = ts.pivot_table(index="date", columns="pass", values="vod")
print(
    f"Daily records: {len(ts_wide):,}  ({ts_wide.index.min().date()} to "
    f"{ts_wide.index.max().date()})"
)
print(ts_wide.notna().sum().to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True, dpi=120)

for ax, pass_name, color in zip(axes, ["asc", "desc"], ["#2196F3", "#FF9800"]):
    vals = ts_wide[pass_name].dropna()
    ax.scatter(vals.index, vals.values, s=0.3, alpha=0.4, color=color, rasterized=True)
    rolling = ts_wide[pass_name].rolling(30, center=True, min_periods=10).mean()
    ax.plot(rolling.index, rolling.values, color="k", lw=0.8, label="30-day mean")
    ax.set_ylabel(f"VOD ({pass_name})")
    ax.set_ylim(0, None)
    ax.legend(loc="upper right", fontsize=8)

axes[0].set_title(f"Daily AMSR VOD at {SITE_ID} (2002-2022)")
axes[1].set_xlabel("Date")

fig.tight_layout()
out = os.path.join(OUT_DIR, "amsr_vod_timeseries.png")
fig.savefig(out, dpi=150)
plt.show()
print("Saved:", os.path.abspath(out))

## 3) Seasons, and this site's climatology

The 20 features are the daily record above, reduced over the day-of-year ranges
`amsr_extract._SEASONS` defines — the same ranges `call_ee.py` uses for the
Earth Engine seasonal composites, so a "summer" VOD feature and a "summer"
Landsat feature cover the same days. Winter wraps the year boundary.

In [ ]:
from swapstress.features.amsr_extract import _SEASONS

print("day-of-year ranges (from swapstress.features.amsr_extract._SEASONS):")
for season, (start, end) in _SEASONS.items():
    wrap = "  (wraps the year boundary)" if start > end else ""
    print(f"  {season:<8s} DOY {start:>3d}-{end:<3d}{wrap}")
print("  annual   every valid day")

if not os.path.exists(VOD_CLIM_PATH):
    raise FileNotFoundError(
        f"{VOD_CLIM_PATH} does not exist. The climatology is not one of the "
        "numbered stages and is not built by default: run the extractor in "
        "section 0 first. Sections 3 onward read what it writes."
    )

vod_clim = pd.read_parquet(VOD_CLIM_PATH)
sample_ids = vod_clim["sample_id"].astype(str)
site_row = vod_clim[sample_ids.str.startswith(f"reesh_{SITE_KEY}")]
vod_cols = [c for c in vod_clim.columns if c.startswith("vod_")]

print(f"\nprecomputed climatology for {SITE_ID}:")
site_row[vod_cols].head(1).T

In [ ]:
season_order = ["winter", "spring", "summer", "autumn", "annual"]
row = site_row.iloc[0]

fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
x = np.arange(len(season_order))
width = 0.35

for i, (pass_name, color) in enumerate([("asc", "#2196F3"), ("desc", "#FF9800")]):
    means = [row[f"vod_{pass_name}_mean_{s}"] for s in season_order]
    stds = [row[f"vod_{pass_name}_stddev_{s}"] for s in season_order]
    ax.bar(
        x + i * width,
        means,
        width,
        yerr=stds,
        label=pass_name,
        color=color,
        alpha=0.85,
        capsize=3,
    )

ax.set_xticks(x + width / 2)
ax.set_xticklabels(season_order)
ax.set_ylabel("VOD (mean +/- stddev)")
ax.set_title(f"Seasonal VOD climatology at {SITE_ID}")
ax.legend()

fig.tight_layout()
plt.show()

## 4) The climatology parquet across all sites

One row per training site, 20 VOD columns. The NaN fractions matter: AMSR ends in
2022 and its footprint is ~25 km, so sites in small or coastal grid cells can
come back empty and will be imputed at train time.

In [ ]:
print(f"Shape: {vod_clim.shape}")
print(f"VOD columns ({len(vod_cols)}): {vod_cols}")

print("\nNaN fraction per column:")
print(vod_clim[vod_cols].isna().mean().round(4).to_string())

print("\nDescriptive statistics:")
vod_clim[vod_cols].describe().round(4).T

## 5) Distributions across training sites

In [ ]:
plot_cols = [
    "vod_asc_mean_summer",
    "vod_desc_mean_summer",
    "vod_asc_stddev_summer",
    "vod_asc_mean_winter",
]

fig, axes = plt.subplots(2, 2, figsize=(10, 7), dpi=120)
for ax, col in zip(axes.flat, plot_cols):
    vals = vod_clim[col].dropna()
    ax.hist(vals, bins=60, color="#607D8B", edgecolor="white", linewidth=0.3)
    ax.set_xlabel(col)
    ax.set_ylabel("Sites")
    ax.set_title(f"{col}  (n={len(vals):,})", fontsize=10)

fig.suptitle("VOD climatology across training sites", fontsize=13)
fig.tight_layout()
out = os.path.join(OUT_DIR, "amsr_vod_distributions.png")
fig.savefig(out, dpi=150)
plt.show()
print("Saved:", os.path.abspath(out))

## 6) Spatial pattern of summer VOD

Joining the climatology back to site coordinates. State outlines come from
`swapstress.figures.basemap`, which resolves the boundaries tree the same way the
source registry resolves the MGRS index.

In [ ]:
import geopandas as gpd

from swapstress.figures import basemap

# Not part of the packaged basemap assets, which are CONUS-only. Point this at a
# world outline of your choosing, or set it to None to draw the global panel bare.
WORLD_SHP = os.path.join(
    basemap.boundaries_root(),
    "boundaries",
    "world_countries",
    "World_Countries_shp.shp",
)


def conus_axis(ax):
    """Frame a lon/lat axis on CONUS with state outlines. Display plumbing.

    The descriptor's map figures project to Albers and style their own axes;
    this is the quick notebook equivalent, so it corrects the aspect by
    latitude rather than projecting.
    """
    basemap.load_conus_states().boundary.plot(ax=ax, color="0.5", linewidth=0.4)
    ax.set_xlim(*basemap.CONUS_LON)
    ax.set_ylim(*basemap.CONUS_LAT)
    ax.set_aspect(1 / np.cos(np.deg2rad(np.mean(basemap.CONUS_LAT))))
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


merged = vod_clim.merge(sites, on="sample_id", how="inner")
col = "vod_asc_mean_summer"
valid_all = merged[merged[col].notna()]
print(f"Sites with coordinates and a summer VOD value: {len(valid_all):,}")

vmin, vmax = valid_all[col].quantile([0.02, 0.98])

fig, (ax_g, ax_c) = plt.subplots(2, 1, figsize=(12, 10), dpi=130)

if WORLD_SHP and os.path.exists(WORLD_SHP):
    gpd.read_file(WORLD_SHP).to_crs(4326).boundary.plot(
        ax=ax_g, color="0.6", linewidth=0.3
    )
ax_g.set_title(f"Summer ascending VOD, all sites (n={len(valid_all):,})")
ax_g.set_xlim(-180, 180)
ax_g.set_ylim(-60, 85)
ax_g.set_xlabel("Longitude")
ax_g.set_ylabel("Latitude")
sc = ax_g.scatter(
    valid_all["lon"],
    valid_all["lat"],
    c=valid_all[col],
    cmap="YlGn",
    s=3,
    alpha=0.7,
    vmin=vmin,
    vmax=vmax,
    rasterized=True,
)
fig.colorbar(sc, ax=ax_g, shrink=0.6, label=col)

conus_axis(ax_c)
conus = valid_all[
    valid_all["lon"].between(*basemap.CONUS_LON)
    & valid_all["lat"].between(*basemap.CONUS_LAT)
]
ax_c.set_title(f"CONUS (n={len(conus):,})")
sc_c = ax_c.scatter(
    conus["lon"],
    conus["lat"],
    c=conus[col],
    cmap="YlGn",
    s=6,
    alpha=0.8,
    vmin=vmin,
    vmax=vmax,
    rasterized=True,
)
fig.colorbar(sc_c, ax=ax_c, shrink=0.7, label=col)

fig.tight_layout(h_pad=1.5)
out = os.path.join(OUT_DIR, "amsr_vod_spatial.png")
fig.savefig(out, dpi=150)
plt.show()
print("Saved:", os.path.abspath(out))